In [1]:
# Install latest version from local files (only necessary after pulling updated version from Git).
# Restart Jupyter kernel when install has finished.
%pip install -e ..

Obtaining file:///home/maarten/Documents/Projects/BioPAP/BioPAP
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for BioPAP (pyproject.toml) ... done
  Created wheel for BioPAP: filename=biopap-0.1.dev103+g06ef61606.d20260218-0.editable-py3-none-any.whl size=3401 sha256=6aefd48bc36558e5802b3bcc9bb696499e9d9b44ee547251c1b8de8464a817f6
  Stored in directory: /tmp/pip-ephem-wheel-cache-4umvb9hr/wheels/a9/55/80/ba98fe05e3d05116737c29a1c667eb5bffec478c135f508c7b
Successfully built BioPAP
  Attempting uninstall: BioPAP
    Found existing installation: BioPAP 0.1.dev103+g06ef61606.d20260217
    Uninstalling BioPAP-0.1.dev103+g06ef61606.d20260217:
      Successfully uninstalled BioPAP-0.1.dev103+g06ef61606.d20260217
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Some Windows installations require explicitly installing mne (required only once; restart Jupyter kernel after installing).
%pip install mne[full] mne_bids

Note: you may need to restart the kernel to use updated packages.


In [2]:
from biopap.pipeline import PipelineStep, Pipeline

from biopap.modules.load_data import LoadData
from biopap.modules.anonymize import Anonymize
from biopap.modules.rereference import Rereference
from biopap.modules.eog_channels import EOGChannels
from biopap.modules.iir_filter import IIRFilter
from biopap.modules.downsample import Downsample
from biopap.modules.segmentation import Segmentation
from biopap.modules.bids import BIDSConversion
from biopap.modules.plot import Plot
from biopap.modules.blink_detection import BlinkDetection
from biopap.modules.artifact_reject import ArtifactReject
from biopap.modules.set_channel_type_and_montage import SetChannelTypeAndMontage

In [3]:
from pathlib import Path
from pprint import pprint

### Define the data and config files

The config file holds the settings for each of the steps of the pipeline we are going to create. They are kept in a separate file to make it easier to use the same settings for multiple runs, for instance when running preprocessing on many files in parallel.

In [4]:
# To avoid issues with local settings and GitHub updates, create a small notebook
# called 'my_config.ipynb' to hold variables that are likely to change locally. 
# In the same folder as this file, there is an example notebook called
# 'example.my_config.ipynb' that can serve as a template.

config_notebook = './my_config.ipynb'

if Path(config_notebook).exists():
    %run {config_notebook}
else:
    data_file = '/biopap/data/example.bdf'
    config_file = './data/pipeline.yaml'

if not Path(config_file).exists:
    print(f"Config file {config_file} does not exist.")

if not Path(data_file).exists:
    print(f"Data file {data_file} does not exist.")

### Loading data

In [5]:
# Read the data file and load it into memory.
load = LoadData(file_path=data_file, verbose='ERROR')

In [ ]:

import mne 

mne.channels.get_builtin_montages(descriptions=True)

### Setting up the pipeline

In [7]:
# Create a new pipeline, loading the raw data and the configuration file.
pipe = Pipeline(data=load.raw, config_file=config_file, verbose='ERROR')

In [ ]:
# Print the pipeline's config attribute to see the settings you loaded.
pprint(pipe.config)

In [ ]:
# Print info of the pipeline's raw attribute to see the loaded data's main info, and display channel details.
print(load.raw.info)
print(load.raw.describe())

### Adding steps to the pipeline
A pipeline is made up of a number of steps that are executed in order. Each step's output becomes the next step's input. The first step starts with the raw data loaded when the pipeline was created. You first add all necessary the steps to the pipeline, without actually running any processing. Only when you are satisfied the pipeline is complete, the entire pipleline is run with one command.

A step simply consists of one of the available processing classes. The specific settings for that class are read from the config file when the step is added. When you need to run the same class twice with different settings, you can use a custom label in the configuration file to keep the different sets of settings apart.

When adding a step, you can tell the pipeline to plot the resulting (processed) data right after that step has finished. The pipeline will pause and the plot will be shown in an interactive window. In the interactive plot you can do lots of stuff with the raw data (e.g. selection bad channels) see https://mne.tools/stable/auto_tutorials/raw/40_visualize_raw.html. **Note** that the pipeline will only continue when the plot window is closed.

To change the pipeline, call the reset method, and re-add the steps in the right order. There is currently no way to change the order of steps in an existing pipeline, or to remove individual steps.

In [ ]:
# Add processing steps in the order they should be run.
pipe.add_step(Anonymize)
pipe.add_step(Downsample)
pipe.add_step(IIRFilter)
pipe.add_step(Plot)

# print pipeline overview, including each step's configuration settings
print(*pipe.steps, sep='\n')

In [ ]:
# When adding multiple steps for same class w/ different settings, use config label to keep them apart.
pipe.add_step(Downsample, cfg_label='Downsample_1000')

print(*pipe.steps, sep='\n')

In [ ]:
# Settings from the config file can also be overridden ad hoc
pipe.add_step(IIRFilter, cfg={'h_freq': 50})

print(*pipe.steps, sep='\n')

In [ ]:
# Call reset to clear the entire pipeline.
pipe.reset()

In [8]:
# Adding steps.
# pipe.add_step(BIDSConversion)
pipe.add_step(Anonymize)
pipe.add_step(Rereference)
pipe.add_step(EOGChannels)
pipe.add_step(IIRFilter)
pipe.add_step(Downsample)
pipe.add_step(BlinkDetection)
pipe.add_step(Segmentation)
pipe.add_step(ArtifactReject)

print(*pipe.steps, sep='\n')

0: Anonymize ← {'daysback': 100, 'experimenter': 'J. Jansen', 'participant_id': 'AB00123'}
1: Rereference ← {'ref_channels': ['EXG1', 'EXG2']}
2: EOGChannels ← {'ref_channels': ['hEOG', 'vEOG'], 'anode_channels': ['EXG3', 'EXG5'], 'cathode_channels': ['EXG4', 'EXG6']}
3: IIRFilter ← {'l_freq': 0.1, 'h_freq': 35}
4: Downsample ← {'sfreq': 500}
5: BlinkDetection ← None
6: Segmentation ← {'stim_channel': 'Status', 'start': -0.2, 'end': 1, 'baseline': [-0.2, 0]}
7: ArtifactReject ← None


### Running the pipeline

In [9]:
# Run the entire pipeline and plot the end results.
pipe.run()

00. Anonymize ← {'daysback': 100, 'experimenter': 'J. Jansen', 'participant_id': 'AB00123'}
Error: 'Anonymize' object has no attribute 'verbose'
01. Rereference ← {'ref_channels': ['EXG1', 'EXG2']}
Error: 'Rereference' object has no attribute 'verbose'
02. EOGChannels ← {'ref_channels': ['hEOG', 'vEOG'], 'anode_channels': ['EXG3', 'EXG5'], 'cathode_channels': ['EXG4', 'EXG6']}
Error: 'EOGChannels' object has no attribute 'verbose'
03. IIRFilter ← {'l_freq': 0.1, 'h_freq': 35}
Error: 'IIRFilter' object has no attribute 'verbose'
04. Downsample ← {'sfreq': 500}
Error: 'Downsample' object has no attribute 'verbose'
05. BlinkDetection ← None
Error: No EOG channel(s) found
06. Segmentation ← {'stim_channel': 'Status', 'start': -0.2, 'end': 1, 'baseline': [-0.2, 0]}
Error: 'Segmentation' object has no attribute 'verbose'
07. ArtifactReject ← None
Error: the input must be an mne.Epochs instance, got <class 'mne.io.edf.edf.RawEDF'> instead.


In [ ]:
print("done")